In [ ]:
!pip install tiktoken datasets
import torch
import torch.nn as nn
import tiktoken
import time
import os
import json
import ssl
import re
import copy
import random
import math
import urllib.request
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import Dataset, IterableDataset, DataLoader, random_split
from datasets import load_dataset, interleave_datasets

# Detect Local vs Colab Environment
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = "/content/drive/MyDrive/ThinkingMachine"
    print("Running on Google Colab. All models will be saved to Drive.")
except ImportError:
    BASE_DIR = "./models"
    print("Running Locally. Saving to ./models")

os.makedirs(BASE_DIR, exist_ok=True)

TINY_TM_CONFIG = {
    "vocab_size": 50257,
    "context_length": 1024,
    "emb_dim": 384,
    "n_heads": 6,
    "n_layers": 6,
    "drop_rate": 0.1,
    "qkv_bias": False
}

TRAINING_CONFIG = {
    "pretraining": {
        "epochs": 1,
        "max_steps": 100000,
        "batch_size": 10,
        "lr": 4e-4,
        "lr_min": 1e-5,  # Ending LR
        "checkpoint_interval": 1000
    },
    "sft": {
        "epochs": 5,
        "batch_size": 8,
        "lr": 5e-5,
        "lr_min": 1e-6,
        "checkpoint_interval": 500
    },
    "grpo": {
        "epochs": 3,
        "batch_size": 8,
        "group_size": 4,
        "lr": 1e-7,
        "lr_min": 1e-8,
        "beta": 0.04
    }
}

# ==========================================
# 2. MODEL ARCHITECTURE
# ==========================================
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))
    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        return self.scale * (x - mean) / torch.sqrt(var + self.eps) + self.shift

class GELU(nn.Module):
    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(torch.sqrt(torch.tensor(2.0 / torch.pi)) * (x + 0.044715 * torch.pow(x, 3))))

class MultiHeadAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n_heads = cfg["n_heads"]
        self.head_dim = cfg["emb_dim"] // cfg["n_heads"]
        self.W_q = nn.Linear(cfg["emb_dim"], cfg["emb_dim"], bias=cfg["qkv_bias"])
        self.W_k = nn.Linear(cfg["emb_dim"], cfg["emb_dim"], bias=cfg["qkv_bias"])
        self.W_v = nn.Linear(cfg["emb_dim"], cfg["emb_dim"], bias=cfg["qkv_bias"])
        self.out_proj = nn.Linear(cfg["emb_dim"], cfg["emb_dim"])
        self.register_buffer("mask", torch.triu(torch.ones(cfg["context_length"], cfg["context_length"]), diagonal=1))

    def forward(self, x):
        b, seq, _ = x.shape
        q = self.W_q(x).view(b, seq, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.W_k(x).view(b, seq, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.W_v(x).view(b, seq, self.n_heads, self.head_dim).transpose(1, 2)
        scores = (q @ k.transpose(-2, -1)) / (self.head_dim ** 0.5)
        scores.masked_fill_(self.mask.bool()[:seq, :seq], -float("inf"))
        weights = torch.softmax(scores, dim=-1)
        return self.out_proj((weights @ v).transpose(1, 2).contiguous().view(b, seq, -1))

class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(cfg)
        self.ff = nn.Sequential(nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]), GELU(), nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]))
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
    def forward(self, x):
        x = x + self.att(self.norm1(x))
        x = x + self.ff(self.norm2(x))
        return x

class ThinkingMachine(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.blocks = nn.Sequential(*[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.xavier_uniform_(module.weight)
            if module.bias is not None: torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx):
        seq = idx.shape[1]
        x = self.tok_emb(idx) + self.pos_emb(torch.arange(seq, device=idx.device))
        x = self.blocks(x)
        return self.out_head(self.final_norm(x))


# --- Training Logic ---
def get_path(filename):
    return os.path.join(BASE_DIR, filename)

def save_checkpoint(model, optimizer, step, epoch, filename):
    full_path = get_path(filename)
    tmp_path = full_path + ".tmp"
    torch.save({
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict() if optimizer else None,
        'step': step,
        'epoch': epoch,
        'rng_state': torch.get_rng_state()
    }, tmp_path)
    if os.path.exists(tmp_path):
        os.replace(tmp_path, full_path)
        print(f"Saved {filename} | Ep {epoch} Step {step}")

def load_checkpoint(model, optimizer, filename):
    full_path = get_path(filename)
    if not os.path.exists(full_path):
        print(f"No checkpoint '{filename}' found. Starting fresh.")
        return 0, 0

    print(f"Resuming from {filename}...")
    ckpt = torch.load(full_path, map_location='cpu')

    model.load_state_dict(ckpt['model_state_dict'])
    if optimizer and ckpt['optimizer_state_dict']:
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])

    return ckpt.get('step', 0), ckpt.get('epoch', 0)

def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0) # remove batch dimension
    return tokenizer.decode(flat.tolist())

def generate_creative(model, tokenizer, prompt, max_tokens, context_size, device, temp=0.8, top_k=10 ):
    model.eval()
    idx = torch.tensor(tokenizer.encode(prompt)).unsqueeze(0).to(device)
    eos_id = tokenizer.encode('<|endoftext|>', allowed_special={'<|endoftext|>'})[0]
    for _ in range(max_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]

        # New: Filter logits with top_k sampling
        if top_k is not None:
            # Keep only top_k values
            top_logits, _ = torch.topk(logits, top_k)
            min_val = top_logits[:, -1]
            logits = torch.where(logits < min_val, torch.tensor(float("-inf")).to(logits.device), logits)

        # New: Apply temperature scaling
        if temp > 0.0:
            logits = logits / temp

            # Apply softmax to get probabilities
            probs = torch.softmax(logits, dim=-1)  # (batch_size, context_len)

            # Sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1)  # (batch_size, 1)

        # get idx of the vocab entry with the highest logits value
        else:
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)  # (batch_size, 1)

        if idx_next == eos_id:  # Stop generating early if end-of-sequence token is encountered and eos_id is specified
            break

        # Same as before: append sampled index to the running sequence
        idx = torch.cat((idx, idx_next), dim=1)  # (batch_size, num_tokens+1)

    return token_ids_to_text(idx,tokenizer)

def create_or_load_val(tokenizer, device):
    val_file = get_path("pretraining_validation_sete.pt")

    if os.path.exists(val_file):
        print("Loading validation set...")
        val_data = torch.load(val_file, map_location='cpu')
        val_set = [(x.to(device), y.to(device)) for x, y in val_data]
        print(f"Loaded {len(val_set)} validation samples")
        return val_set

    print("Creating validation set from first 100 samples...")
    val_samples = []

    # First 100 from TinyStories
    print("  - Taking first 50 from TinyStories...")
    ds_tiny = load_dataset("roneneldan/TinyStories", split="train", streaming=True)
    for i, item in enumerate(ds_tiny):
        if i >= 100:
            break
        tokens = tokenizer.encode(item['text'], allowed_special={'<|endoftext|>'})[:1024]
        if len(tokens) > 1:
            val_samples.append((
                torch.tensor(tokens[:-1]),
                torch.tensor(tokens[1:])
            ))

    # First 100 from FineWeb
    print("  - Taking first 50 from FineWeb-edu...")
    ds_fineweb = load_dataset("HuggingFaceFW/fineweb-edu", name="sample-10BT", split="train", streaming=True)
    for i, item in enumerate(ds_fineweb):
        if i >= 100:
            break
        tokens = tokenizer.encode(item['text'], allowed_special={'<|endoftext|>'})[:1024]
        if len(tokens) > 1:
            val_samples.append((
                torch.tensor(tokens[:-1]),
                torch.tensor(tokens[1:])
            ))

    # Save to disk
    print(f"Saving {len(val_samples)} validation samples...")
    torch.save(val_samples, val_file)

    val_set = [(x.to(device), y.to(device)) for x, y in val_samples]
    print(f"Created validation set: {len(val_set)} samples")
    return val_set

def validate(model, val_set):
  model.eval()
  total_loss = 0
  with torch.no_grad():
      for x, y in val_set:
          logits = model(x.unsqueeze(0))
          loss = nn.functional.cross_entropy(logits.flatten(0,1), y.unsqueeze(0).flatten())
          total_loss += loss.item()
  model.train()
  return total_loss / len(val_set)

def run_pretraining():
    torch.cuda.empty_cache()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\nPHASE 1: Pre-training on {device}")

    cfg = TRAINING_CONFIG["pretraining"]
    tokenizer = tiktoken.get_encoding("gpt2")
    model = ThinkingMachine(TINY_TM_CONFIG).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["lr"])

    # 1. Resume
    if os.path.exists(get_path("pretrained_model.pth")):
        print("Already completed. Skipping.")
        return
    start_step, start_epoch = load_checkpoint(model, optimizer, "pretrain_latest.pth")

    steps_per_epoch = cfg["max_steps"]
    total_steps = steps_per_epoch * cfg["epochs"]
    remaining_steps = total_steps - (start_epoch * steps_per_epoch) - start_step

    scheduler = CosineAnnealingLR(
        optimizer,
        T_max=max(1, remaining_steps), # Prevent division by zero
        eta_min=cfg["lr_min"]
    )

    # Create validation set FIRST
    val_set = create_or_load_val(tokenizer, device)

    # 2. Unified Dataset (Story + Logic)
    ds1 = load_dataset("roneneldan/TinyStories", split="train", streaming=True).skip(100)
    ds2 = load_dataset("HuggingFaceFW/fineweb-edu", name="sample-10BT", split="train", streaming=True).skip(100)
    mixed_ds = interleave_datasets([ds1, ds2], probabilities=[0.5, 0.5])

    skip_count = (start_epoch * cfg["max_steps"] * cfg["batch_size"]) + (start_step * cfg["batch_size"])
    if skip_count > 0:
        print(f"Skipping {skip_count} samples to resume position...")
        mixed_ds = mixed_ds.skip(skip_count)

    class PackedDS(IterableDataset):
        def __iter__(self):
            buf = []
            for item in mixed_ds:
                tokens = tokenizer.encode(item['text'], allowed_special={'<|endoftext|>'}) + [50256]
                buf.extend(tokens)
                while len(buf) >= 1025:
                    yield torch.tensor(buf[:1024]), torch.tensor(buf[1:1025])
                    buf = buf[1024:]

    loader = DataLoader(PackedDS(), batch_size=cfg["batch_size"])
    eval_prompt = "One day, the sun decided not to rise. The people were  "
    print(f"\nEval test Before training: {generate_creative(model, tokenizer, eval_prompt, 150,  TINY_TM_CONFIG["context_length"], device, 1.2, 40)}\n")
    val_loss = validate(model, val_set)
    print(f"\nValidation Loss @ start of training: {val_loss:.4f}")
    start_time = time.time()
    model.train()
    total_loss = 0
    loss_history = []  # Track recent losses
    try:
        step = start_step
        for epoch in range(start_epoch, cfg["epochs"]):

            for x, y in loader:
                if step >= cfg["max_steps"]: break
                model.train()
                x, y = x.to(device), y.to(device)
                optimizer.zero_grad()
                loss = nn.functional.cross_entropy(model(x).flatten(0,1), y.flatten())
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
                total_loss += loss.item()

                loss_history.append(loss.item())
                if len(loss_history) > 1000:  # Rolling average of last 1000 steps
                  loss_history.pop(0)

                if step % 100 == 0 and step > 0:
                  avg_loss = sum(loss_history) / len(loss_history)
                  val_loss = validate(model, val_set)
                  current_lr = scheduler.get_last_lr()[0]
                  print(f"[{time.strftime('%H:%M:%S')}] | Ep {epoch} Step {step} | val_loss : {val_loss:.4f} | Current Loss: {loss.item():.4f} | LR: {current_lr:.2e}")

                if step % cfg["checkpoint_interval"] == 0 and step > 0:
                    save_checkpoint(model, optimizer, step, epoch, "pretrain_latest.pth")

                if step % 5000 == 0 and step > 0:
                  print(f"\nEval TEST: {generate_creative(model, tokenizer, eval_prompt, 150, TINY_TM_CONFIG["context_length"], device, 1, 20)}\n")

                step += 1

            # Epoch complete
            step = 0 # Reset step counter for next epoch
            save_checkpoint(model, optimizer, 0, epoch+1, f"pretrain_latest_{epoch}.pth")
            print(f"\nEval TEST post epoch : {epoch}: {generate_creative(model, tokenizer, eval_prompt, 150, TINY_TM_CONFIG["context_length"], device, 1, 40)}\n")

    except KeyboardInterrupt:
        print("Stopped.")
        return

    save_checkpoint(model, optimizer, 0, 0, "pretrained_model.pth")
    print(f"Pre-training Complete! Total time: {(time.time() - start_time) / 3600:.2f} hours")

# ==========================================
# PHASE 2: SFT (Train/Test/Val Split)
# ==========================================
def load_sft_data():
  f_train = get_path("sft_train.json")
  f_val = get_path("sft_val.json")

  # 1. Data Loading (Check Cache -> Download -> Save)
  if os.path.exists(f_train) and os.path.exists(f_val):
      print("Found cached SFT data in Drive. Loading...")
      with open(f_train, 'r') as f: train_data = json.load(f)
      with open(f_val, 'r') as f: val_data = json.load(f)
  else:
      print("No cached SFT data found. Processing and Saving...")
      data = []

      # Alpaca
      if not os.path.exists("alpaca.json"):
          urllib.request.urlretrieve("https://raw.githubusercontent.com/tatsu-lab/stanford_alpaca/main/alpaca_data.json", "alpaca.json")
      with open("alpaca.json") as f:
          for item in json.load(f):
              p = f"Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n### Instruction:\n{item['instruction']}\n\n### Response:\n"
              data.append({"prompt": p, "full": p + item['output'] + "<|endoftext|>"})

      # GSM8k
      gsm = load_dataset("gsm8k", "main", split="train")
      for item in gsm:
          ans = item['answer'].split('####')
          p = f"Question: {item['question']}\nSolve step by step.\n<think>\n"
          full = f"{p}{ans[0].strip()}\n</think>\nAnswer: {ans[1].strip()}<|endoftext|>"
          data.append({"prompt": p, "full": full})

      # OpenOrca (Limited)
      orca = load_dataset("Open-Orca/OpenOrca", split="train", streaming=True)
      for i, item in enumerate(orca):
          if i >= 30000: break # Reduced slightly for speed
          p = f"{item['system_prompt']}\n\n{item['question']}\n\n"
          data.append({"prompt": p, "full": p + item['response'] + "<|endoftext|>"})

      # Split & Save
      total = len(data)
      train_size = int(0.95 * total)
      val_size = total - train_size
      train_data, val_data = random_split(data, [train_size, val_size])

      # Convert subsets back to lists for JSON dumping
      train_data = list(train_data)
      val_data = list(val_data)

      with open(f_train, 'w') as f: json.dump(train_data, f, indent=2)
      with open(f_val, 'w') as f: json.dump(val_data, f, indent=2)
      print("Saved SFT datasets to JSON in Drive.")

      print(f"Data Split: Train {len(train_data)} | Val {len(val_data)}")

  return train_data, val_data

def test_loader(model, test_loader):
  # Run Validation
  model.eval()
  loss = 0
  with torch.no_grad():
      for vx, vy in test_loader:
          loss += nn.functional.cross_entropy(model(vx).flatten(0,1), vy.flatten()).item()
  model.train()
  return loss/len(test_loader)

def run_sft():
    torch.cuda.empty_cache()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n🏫 PHASE 2: SFT on {device}")

    cfg = TRAINING_CONFIG["sft"]
    if os.path.exists(get_path("sft_model.pth")):
        print("SFT already finished. Skipping.")
        return

    tokenizer = tiktoken.get_encoding("gpt2")

    train_data, val_data = load_sft_data()

    class SFTDS(Dataset):
        def __init__(self, data): self.data = data
        def __len__(self): return len(self.data)
        def __getitem__(self, idx):
            item = self.data[idx]
            return {
                "ids": tokenizer.encode(item['full'], allowed_special={'<|endoftext|>'}),
                "p_len": len(tokenizer.encode(item['prompt'], allowed_special={'<|endoftext|>'}))
            }

    def collate(batch):
        max_len = min(1024, max([len(x['ids']) for x in batch]))
        seq_len = max(1, max_len - 1)
        inputs = torch.full((len(batch), seq_len), 50256, dtype=torch.long, device=device)
        targets = torch.full((len(batch), seq_len), -100, dtype=torch.long, device=device)
        for i, item in enumerate(batch):
            ids = torch.tensor(item['ids'])[:max_len]
            if len(ids) < 2: continue
            inputs[i, :len(ids)-1] = ids[:-1]
            targets[i, :len(ids)-1] = ids[1:]
            mask_end = max(0, min(item['p_len'] - 1, len(ids)-1))
            targets[i, :mask_end] = -100
        return inputs, targets

    train_loader = DataLoader(SFTDS(train_data), batch_size=cfg["batch_size"], shuffle=True, collate_fn=collate)
    val_loader = DataLoader(SFTDS(val_data), batch_size=cfg["batch_size"], collate_fn=collate)

    # 3. Model & Resume
    model = ThinkingMachine(TINY_TM_CONFIG).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["lr"])

    if os.path.exists(get_path("sft_latest.pth")):
        start_step, start_epoch = load_checkpoint(model, optimizer, "sft_latest.pth")
    else:
        load_checkpoint(model, optimizer, "pretrained_for_sft_model.pth")
        start_step, start_epoch = 0, 0

    steps_per_epoch = len(train_loader)
    total_steps = steps_per_epoch * cfg["epochs"]
    remaining_steps = total_steps - (start_epoch * steps_per_epoch) - start_step

    scheduler = CosineAnnealingLR(
        optimizer,
        T_max=max(1, remaining_steps), # Prevent division by zero
        eta_min=cfg["lr_min"]
    )

    # 4. Train Loop
    start_time = time.time()
    for epoch in range(start_epoch, cfg["epochs"]):
        total_loss = 0
        for step, (x, y) in enumerate(train_loader):
            # We simply check if step < start_step.
            if step < start_step: continue
            model.train()
            optimizer.zero_grad()
            loss = nn.functional.cross_entropy(model(x).flatten(0,1), y.flatten())
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()
            scheduler.step()
            if step % 100 == 0 and step > 0:
                avg_loss = total_loss/step
                val_loss = test_loader(model, val_loader)
                curr_lr = scheduler.get_last_lr()[0]
                print(f"[{time.strftime('%H:%M:%S')}] | SFT Ep {epoch} Step {step}| val_loss : {val_loss:.4f} | cur loss {loss.item():.4f}  | avg loss {avg_loss:.4f} | LR: {curr_lr:.2e}")
            if step % cfg["checkpoint_interval"] == 0 and step > 0:
                 save_checkpoint(model, optimizer, step, epoch, "sft_latest.pth")

        # End of Epoch
        start_step = 0 # Reset for next epoch
        save_checkpoint(model, optimizer, 0, epoch+1, "sft_latest.pth")

    save_checkpoint(model, optimizer, 0, 0, "sft_model.pth")
    print(f"SFT Phase Complete.! Total time: {(time.time() - start_time) / 3600:.2f} hours")


# ==========================================
# PHASE 3: GRPO
# ==========================================
def load_rl_data():

  f_rl_data = get_path("grpo_data.json")

  train_data = []
  if os.path.exists(f_rl_data):
        print("Found cached GRPO data. Loading...")
        with open(f_rl_data, 'r') as f: train_data = json.load(f)
  else:
      print("No cached GRPO data. Processing...")
      gsm = load_dataset("gsm8k", "main", split="train")

      for item in gsm:
          p = f"Question: {item['question']}\nSolve step by step.\n<think>\n"
          ans = item['answer'].split('####')[-1].strip()

          train_data.append({"prompt": p, "answer": ans})

      with open(f_rl_data, 'w') as f: json.dump(train_data, f, indent=2)
      print(f"Saved {len(train_data)} RL samples to Drive.")
  return train_data

def run_grpo():
    cfg = TRAINING_CONFIG["grpo"]
    tokenizer = tiktoken.get_encoding("gpt2")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n GRPO on {device}")

    cfg = TRAINING_CONFIG["grpo"]
    tokenizer = tiktoken.get_encoding("gpt2")

    # Prep RL Data
    train_data = load_rl_data()

    # Model
    model = ThinkingMachine(TINY_TM_CONFIG).to(device)

    if os.path.exists(get_path("grpo_latest.pth")):
         start_step, start_epoch = load_checkpoint(model, None, "grpo_latest.pth")
    else:
         load_checkpoint(model, None, "sft_model.pth")
         start_epoch, start_step = 0, 0

    ref_model = copy.deepcopy(model).eval()
    for p in ref_model.parameters(): p.requires_grad = False
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["lr"])
    steps_per_epoch = len(train_data) // cfg["batch_size"]
    total_steps = steps_per_epoch * cfg["epochs"]
    remaining_steps = total_steps - (start_epoch * steps_per_epoch) - (start_step // cfg["batch_size"])

    scheduler = CosineAnnealingLR(
        optimizer,
        T_max=max(1, remaining_steps),
        eta_min=cfg["lr_min"]
    )

    def get_rewards(texts, answers):
        rewards = []

        for text, ans in zip(texts, answers):
            score = 0.0

            think_content = ""
            if "<think>" in text and "</think>" in text:
                think_content = text.split("<think>")[1].split("</think>")[0].strip()
                score += 0.2  # small reward for valid tags

            if "Answer:" in text:
                answer_text = text.split("Answer:")[-1].strip()
                score += 0.1  # small reward for valid format
            else:
                answer_text = text.strip()


            nums = re.findall(r"[-+]?\d*\.\d+|\d+", answer_text)
            correctness_reward = 0.0
            if nums:
                try:
                    expected_str = str(ans).split('####')[-1].strip()
                    expected = float(re.findall(r"[-+]?\d*\.\d+|\d+", expected_str)[-1])
                    predicted = float(nums[-1])

                    if abs(predicted - expected) < 0.01:
                        correctness_reward = 2.0 #  reward for right answers
                    elif abs(predicted - expected) / max(abs(expected), 1e-8) < 0.1:
                        correctness_reward = 0.5 
                except Exception:
                    pass

            score += correctness_reward

            # If the think block has no math operators (+, -, *, /, =), it's probably garbage text.
            math_ops = ['+', '-', '*', '/', '=']
            if think_content:
                if not any(op in think_content for op in math_ops):
                    score -= 0.5 # Penalty for yapping without math
                else:
                    score += 0.1 # Small reward for attempting math

            # Step-by-Step Reward. A "valid step" has words and numbers.
            if think_content:
                lines = [l.strip() for l in think_content.split('\n') if l.strip()]
                valid_steps = 0
                for line in lines:
                    # Check if line has at least 3 words AND at least 1 digit
                    if len(line.split()) >= 3 and any(c.isdigit() for c in line):
                        valid_steps += 1

                # Reward up to 5 valid steps (0.1 each)
                score += min(valid_steps * 0.1, 0.5)

                #  Penalty for duplicates
                # If the number of unique lines is less than total lines, it's repeating.
                if len(set(lines)) < len(lines):
                    score -= 0.5

                # If the Correct Answer is '8' and model predicted '8',
                # but the think trace has NO match with the answer, it might be lucky.
                # (This forces the reasoning to "land" on the answer)
                if correctness_reward > 1.0:
                    # Check if the number '8' actually appears in the reasoning trace
                    # This penalizes: <think>2+2=4</think> Answer: 8 (Lucky guess)
                    if str(int(predicted)) not in think_content:
                        score -= 0.5 # "Show your work" penalty

            rewards.append(score)

        return torch.tensor(rewards).to(device)

    def get_logprobs(m, ids):
        return torch.gather(nn.functional.log_softmax(m(ids), -1), -1, ids.unsqueeze(-1)).squeeze(-1)

    test_q = "Question: If it takes 1 hour to dry 1 shirt outside in the sun, how long does it take to dry 5 shirts?"
    # Format it exactly like training data
    test_prompt = f"{test_q}\nSolve step by step.\n<think>\n"
    print(f"\nRL Testing before training: {test_q}")
    print(generate_creative(model, tokenizer, test_prompt, 150, TINY_TM_CONFIG["context_length"], device, 0.8))
    print("-" * 30)

    print("Starting GRPO...")

    start_time = time.time()
    for epoch in range(start_epoch, cfg["epochs"]):
        # Batching manually for control
        total_loss = 0
        for i in range(0, len(train_data), cfg["batch_size"]):
            # Resume skip
            if i < start_step: continue
            model.train()
            batch = train_data[i:i+cfg["batch_size"]]
            prompts, answers = zip(*batch)
            # Clear CUDA cache before generation
            if i % 100 == 0:
                torch.cuda.empty_cache()
                torch.cuda.synchronize()
            # Generate Group
            completions, tokens_list = [], []
            for p in prompts:
                p_ids = torch.tensor(tokenizer.encode(p)).to(device)

                for group_idx in range(cfg["group_size"]):
                    curr = p_ids.unsqueeze(0)

                    for step in range(150):
                        try:
                            with torch.no_grad():
                               
                                if curr.shape[1] >= 1024:
                                    print(f"Sequence reached max length, stopping")
                                    break
                              
                                # Ensure curr doesn't exceed context length
                                idx_cond = curr[:, -1024:]

                                # VALIDATE INPUT TOKENS
                                if (idx_cond >= TINY_TM_CONFIG["vocab_size"]).any():
                                    print(f"ERROR at step {step}: curr contains invalid token IDs")
                                    print(f"Invalid tokens: {idx_cond[idx_cond >= TINY_TM_CONFIG['vocab_size']]}")
                                    print(f"Max token ID: {idx_cond.max().item()}")
                                    break

                                logits = model(idx_cond)[:, -1, :]

                                probs = torch.softmax(logits, -1)
                                next_tok = torch.multinomial(probs, 1)

                                next_tok_val = next_tok.item()
                                if next_tok_val == 50256:  # EOS
                                    break

                                curr = torch.cat((curr, next_tok), 1)

                        except Exception as e:
                            print(f"Exception during generation at step {step}: {e}")
                            print(f"curr shape: {curr.shape}")
                            print(f"curr max token: {curr.max().item()}")
                            break

                    tokens_list.append(curr[0])

                    try:
                        decoded = tokenizer.decode(curr[0].tolist())
                        completions.append(decoded)
                    except Exception as e:
                        print(f"Decoding error: {e}")
                        completions.append("")  # Add empty string to maintain alignment

            # Pad & Mask
            max_len = max([len(t) for t in tokens_list])
            padded = torch.full((len(tokens_list), max_len), 50256, device=device)
            mask = torch.zeros_like(padded)
            for j, t in enumerate(tokens_list):
                padded[j, :len(t)] = t
                mask[j, 20:len(t)] = 1.0

            # Loss
            expanded_ans = [a for a in answers for _ in range(cfg["group_size"])]
            rewards = get_rewards(completions, expanded_ans)
            adv = (rewards - rewards.mean()) / (rewards.std() + 1e-8)

            log_probs = get_logprobs(model, padded)
            with torch.no_grad(): ref_log_probs = get_logprobs(ref_model, padded)

            ratio = torch.exp(log_probs - ref_log_probs)
            kl = log_probs - ref_log_probs
            loss = -((ratio * adv.unsqueeze(1) - cfg["beta"] * kl) * mask).sum() / mask.sum()

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            total_loss += loss.item()
            if i % 100 == 0 and i > 0:
              avg_loss = total_loss/i
              curr_lr = scheduler.get_last_lr()[0]
              print(f"[{time.strftime('%H:%M:%S')}] | GRPO Ep {epoch} Step {i} | Current Loss {loss.item():.4f}  | Avg Loss {avg_loss:.4f} | LR: {curr_lr:.2e}")

            # Save step as the index 'i'
            if i % 1000 == 0 and i > 0:
                  save_checkpoint(model, optimizer, i, epoch, "grpo_latest.pth")
                  print(f"\nRL Checkup: {test_q}")
                  print(generate_creative(model, tokenizer, test_prompt, 150, TINY_TM_CONFIG["context_length"], device, 1.2, 10))
                  print("-" * 30)

        start_step = 0
        save_checkpoint(model, optimizer, 0, epoch+1, "grpo_latest.pth")

    save_checkpoint(model, optimizer, 0, 0, "reasoning_model_grpo.pth")
    print(f"GRPO Phase Complete.! Total time: {(time.time() - start_time) / 3600:.2f} hours")


if __name__ == "__main__":
    run_pretraining()
    run_sft()
    run_grpo()

Starting Mixed Healing (1024 Context) on: cpu
Performing Surgery on ./models\pretrained.pth...
✅ Surgery Complete
Mixing TinyStories (Creativity) + FineWeb-Edu (Logic)...


Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

Healing started... for epoch: 0 step : 0


: 